# 로컬 트레이싱과 지연

Python 3.12 새 커널에서 RAG 단계별 로컬 실행 추적 준비부터 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** trace는 한 요청의 전체 실행이고 span은 검색·프롬프트·생성처럼 그 안의 한 단계입니다. 각 span에 시작·종료 시각, 입력·출력 요약, 토큰 사용량을 남기면 전체 지연에서 어느 단계가 차지한 시간을 계산할 수 있습니다. 첫 토큰 도착 시간과 전체 완료 시간은 서로 다른 사용자 경험을 나타냅니다.

**다른 상황에 적용.** 검색은 40ms인데 생성이 2초인 요청과 검색이 1.5초인데 생성이 600ms인 요청은 총시간이 비슷해도 수정할 부분이 다릅니다. 단계별 span을 남겨야 모델 교체와 인덱스 조정 중 무엇을 시험할지 정할 수 있습니다.

**비교와 경계.** 이 노트북의 고정 시간값은 추적 구조와 계산법을 배우는 합성 데이터입니다. 실제 네트워크 병목을 측정한 결과가 아닙니다. 평균 지연만 보면 드문 장기 지연을 놓치므로 다음 절의 백분위수와 함께 봅니다.


## Python 3.12 패키지 버전 고정

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## RAG 단계별 로컬 실행 추적 준비

**이 셀의 코드 읽기.** 이 셀은 한 요청의 시작·종료 시각과 children에 담을 단계별 시각을 고정된 합성값으로 새로 만듭니다. 실제 요청을 실행해 측정한 시간이 아니며, 다음 셀에서 기록 구조와 소요 시간 계산을 관찰하기 위한 자료입니다.


In [2]:
# 한 대화의 재작성·검색·리랭킹·생성 구간을 밀리초 단위로 기록합니다.
trace = {
 'trace_id': 'local-trace-001', 'conversation_id': 'conv-001',
 'started_ms': 0, 'first_token_ms': 146, 'ended_ms': 301,
 'children': [
  {'name': 'rewrite', 'start': 0, 'end': 12},
  {'name': 'retrieve', 'start': 12, 'end': 43},
  {'name': 'rerank', 'start': 43, 'end': 61},
  {'name': 'generate', 'start': 61, 'end': 301},
 ],
}

## 첫 토큰 도착 시간과 전체 지연 계산

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 생성 시작 시각을 기준으로 TTFT를, 요청 시작을 기준으로 total을 계산합니다.
generate = next(x for x in trace['children'] if x['name'] == 'generate')
ttft = trace['first_token_ms'] - generate['start']
total = trace['ended_ms'] - trace['started_ms']
print('TTFT(ms):', ttft)
print('total(ms):', total)
for run in trace['children']:
    print(run['name'], run['end'] - run['start'])

TTFT(ms): 85
total(ms): 301
rewrite 12
retrieve 31
rerank 18
generate 240


## 캐시 읽기·생성 사용량 분리

**이 셀의 코드 읽기.** `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# 입출력 토큰과 캐시 read·creation 토큰을 서로 다른 필드로 보존합니다.
usage = {
 'input_tokens': 1200, 'output_tokens': 180,
 'cache_read_tokens': 420, 'cache_creation_tokens': 80,
}
trace['usage'] = usage
print(usage)
assert ttft == 85 and total == 301
print('cloud 업로드: 미실행(LangSmith API 키 없음)')

{'input_tokens': 1200, 'output_tokens': 180, 'cache_read_tokens': 420, 'cache_creation_tokens': 80}
cloud 업로드: 미실행(LangSmith API 키 없음)
